In [0]:
%sql
USE CATALOG fraud_detection;
USE SCHEMA gold;

CREATE VOLUME chk_volume;

In [0]:
from pyspark.sql.functions import *

gold_df = spark.readStream.table("fraud_detection.silver.transactions_silver") \
    .withColumn(
        "fraud_flag",
        (col("is_high_amount") == True) |
        (col("is_night_txn") == True)
    )

gold_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/fraud_detection/gold/chk_volume/transactions_chk_fin") \
    .trigger(availableNow=True) \
    .toTable("transactions_gold")

In [0]:
%sql
select count(*) from transactions_gold